In [1]:
!pip install -q wandb scikit-learn pandas matplotlib joblib

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

import joblib
import wandb


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
wandb.login()

wandb: WARNING Failed to create global config settings in: /Users/DELL/.config/wandb. Settings will not be persisted.
wandb: Currently logged in as: kothariabhishek091 (kothariabhishek091-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
# Download the Dermatology dataset from UCI (if not already present)
import urllib.request
data_path = "dermatology.data"
data_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data"

if not os.path.exists(data_path):
    print("Downloading dermatology.data ...")
    urllib.request.urlretrieve(data_url, data_path)
    print("Download complete.")
else:
    print("dermatology.data already exists")

Download complete.


In [12]:
import os
print("Exists:", os.path.exists("dermatology.data"))

Exists: True


In [14]:
def load_dermatology_data(path: str = "dermatology.data", test_size: float = 0.3, random_state: int = 42):
    """
    Load the UCI Dermatology dataset, handle missing values, and return train/validation splits.
    
    Returns
    -------
    X_train, X_val, y_train, y_val, feature_names, class_names
    """
    # Dataset has no header in the raw file, so we just give generic column names
    # According to the original lab, there are 34 features + 1 label column.
    n_cols = 35
    col_names = [f"f{i}" for i in range(n_cols - 1)] + ["label"]
    
    df = pd.read_csv(
        path,
        header=None,
        names=col_names,
        na_values="?",
    )

    # Very small number of missing values: fill them with column median
    df = df.fillna(df.median(numeric_only=True))

    # Features and target
    X = df.drop("label", axis=1)
    y = df["label"].astype(int) - 1  # labels start at 1; make them 0..num_classes-1

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    feature_names = list(X.columns)
    class_names = sorted(y.unique())

    return X_train, X_val, y_train, y_val, feature_names, class_names


# Quick sanity check
X_train, X_val, y_train, y_val, feature_names, class_names = load_dermatology_data()
print("Train shape:", X_train.shape, "Val shape:", X_val.shape)
print("Classes:", class_names)

Train shape: (256, 34) Val shape: (110, 34)
Classes: [0, 1, 2, 3, 4, 5]


In [21]:
def train_random_forest(config=None):
    """
    Train a Random Forest classifier and log everything to W&B.
    
    This function is compatible with W&B sweeps: it takes a `config` dict,
    initializes a run, and uses config values as hyperparameters.
    """
    with wandb.init(
        project="Lab1-visualize-models",
        job_type="train",
        config=config,
    ) as run:
        cfg = wandb.config

        # Load data
        X_train, X_val, y_train, y_val, feature_names, class_names = load_dermatology_data(
            test_size=cfg.test_size,
            random_state=cfg.random_state,
        )

        # Define model from config
        model = RandomForestClassifier(
            n_estimators=cfg.n_estimators,
            max_depth=cfg.max_depth,
            min_samples_split=cfg.min_samples_split,
            min_samples_leaf=cfg.min_samples_leaf,
            max_features=cfg.max_features,
            random_state=cfg.random_state,
            n_jobs=-1,
        )

        # Train
        model.fit(X_train, y_train)

        # Predict
        y_pred = model.predict(X_val)
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_val)
        else:
            y_proba = None

        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1_macro = f1_score(y_val, y_pred, average="macro")
        report = classification_report(y_val, y_pred, output_dict=True)

        wandb.log({
            "val_accuracy": acc,
            "val_f1_macro": f1_macro,
        })

        # Optional ROC AUC for multi-class (one-vs-rest)
        if y_proba is not None:
            try:
                roc_auc_ovr = roc_auc_score(y_val, y_proba, multi_class="ovr")
                wandb.log({"val_roc_auc_ovr": roc_auc_ovr})
            except Exception as e:
                print("ROC AUC could not be computed:", e)

        # Log per-class F1 scores
        for cls in class_names:
            cls_key = str(cls)
            if cls_key in report:
                wandb.log({f"f1_class_{cls_key}": report[cls_key]["f1-score"]})
        # Confusion matrix 
        y_true = np.asarray(y_val, dtype=int)
        y_pred_arr = np.asarray(y_pred, dtype=int)
        class_names = sorted(np.unique(y_true))  # ints 0..5

        wandb.log({
            "confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_true,
            preds=y_pred_arr,
            class_names=list(class_names),   # ints, not strings
            )
        })

        # Feature importance
        importances = model.feature_importances_
        fi_df = pd.DataFrame(
            {"feature": feature_names, "importance": importances}
        ).sort_values("importance", ascending=False)

        fi_table = wandb.Table(dataframe=fi_df)
        wandb.log({
            "feature_importance": wandb.plot.bar(
                fi_table, "feature", "importance", title="Random Forest Feature Importance"
            )
        })

        # Sample predictions table
        pred_table = wandb.Table(columns=["index", "true_label", "pred_label"])
        for i in range(min(len(y_val), 50)):
            pred_table.add_data(int(i), int(y_val.iloc[i]), int(y_pred[i]))
        wandb.log({"sample_predictions": pred_table})

        # Save model locally
        os.makedirs("models", exist_ok=True)
        model_path = os.path.join("models", "rf_dermatology.pkl")
        joblib.dump(model, model_path)

        # Log as W&B model artifact
        artifact = wandb.Artifact(
            name="rf-dermatology-model",
            type="model",
            description="Random Forest trained on UCI Dermatology dataset",
            metadata=dict(cfg),
        )
        artifact.add_file(model_path)
        run.log_artifact(artifact)

        # Summaries for easy comparison
        run.summary["val_accuracy"] = acc
        run.summary["val_f1_macro"] = f1_macro

        print(f"[run {run.name}] accuracy={acc:.4f}, f1_macro={f1_macro:.4f}")

## Baseline experiment

First we run a single **baseline** Random Forest experiment with a fixed set of hyperparameters.  
This gives us a reference point before we start the W&B hyperparameter sweep.

In [24]:
baseline_config = {
    "n_estimators": 200,
    "max_depth": 8,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "test_size": 0.3,
    "random_state": 42,
}

train_random_forest(config=baseline_config)

[run clear-energy-2] accuracy=0.9818, f1_macro=0.9796


f1_class_0,▁
f1_class_1,▁
f1_class_2,▁
f1_class_3,▁
f1_class_4,▁
f1_class_5,▁
val_accuracy,▁
val_f1_macro,▁
val_roc_auc_ovr,▁
f1_class_0,1
f1_class_1,0.94444


## Hyperparameter sweep with W&B

Now we define a **W&B Sweep** to explore different Random Forest configurations.

We will vary:

- `n_estimators`  
- `max_depth`  
- `min_samples_split`  
- `min_samples_leaf`  
- `max_features`  

The sweep will try multiple combinations and log:

- validation accuracy
- macro F1-score
- per-class F1 scores
- confusion matrices
- feature importance
- the trained model artifact for each run

Later we can inspect the sweep in the W&B UI and see which configuration performs best.

In [27]:
sweep_config = {
    "method": "bayes",  # can be "grid", "random", or "bayes"
    "metric": {
        "name": "val_f1_macro",
        "goal": "maximize",
    },
    "parameters": {
        "n_estimators": {
            "values": [100, 200, 300, 400]
        },
        "max_depth": {
            "values": [4, 6, 8, None]  # None means "unlimited" in sklearn
        },
        "min_samples_split": {
            "values": [2, 4, 6]
        },
        "min_samples_leaf": {
            "values": [1, 2, 3]
        },
        "max_features": {
            "values": ["sqrt", "log2", None]
        },
        "test_size": {
            "value": 0.3
        },
        "random_state": {
            "value": 42
        },
    },
}

sweep_id = wandb.sweep(
    sweep=sweep_config,
    project="Lab1-visualize-models",
)
print("Created sweep with ID:", sweep_id)

Create sweep with ID: sju09zkz
Sweep URL: https://wandb.ai/kothariabhishek091-northeastern-university/Lab1-visualize-models/sweeps/sju09zkz
Created sweep with ID: sju09zkz


In [29]:
# This will launch multiple runs in the same process.
wandb.agent(
    sweep_id,
    function=train_random_forest,
    count=8,  # number of experiments
)

wandb: Agent Starting Run: voajza3r with config:
wandb: 	max_depth: 4
wandb: 	max_features: None
wandb: 	min_samples_leaf: 3
wandb: 	min_samples_split: 6
wandb: 	n_estimators: 400
wandb: 	random_state: 42
wandb: 	test_size: 0.3


/Users/DELL/anaconda3/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/DELL/anaconda3/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/DELL/anaconda3/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

[run exalted-sweep-1] accuracy=0.8636, f1_macro=0.7351


f1_class_0,▁
f1_class_1,▁
f1_class_2,▁
f1_class_3,▁
f1_class_4,▁
f1_class_5,▁
val_accuracy,▁
val_f1_macro,▁
val_roc_auc_ovr,▁
f1_class_0,0.94118
f1_class_1,0.73913


wandb: Agent Starting Run: v14nlhe1 with config:
wandb: 	max_depth: 4
wandb: 	max_features: log2
wandb: 	min_samples_leaf: 1
wandb: 	min_samples_split: 4
wandb: 	n_estimators: 200
wandb: 	random_state: 42
wandb: 	test_size: 0.3


...

[run earthy-sweep-7] accuracy=0.9091, f1_macro=0.9002


f1_class_0,▁
f1_class_1,▁
f1_class_2,▁
f1_class_3,▁
f1_class_4,▁
f1_class_5,▁
val_accuracy,▁
val_f1_macro,▁
val_roc_auc_ovr,▁
f1_class_0,0.94118
f1_class_1,0.85


wandb: Agent Starting Run: lljtwf92 with config:
wandb: 	max_depth: 6
wandb: 	max_features: log2
wandb: 	min_samples_leaf: 3
wandb: 	min_samples_split: 6
wandb: 	n_estimators: 200
wandb: 	random_state: 42
wandb: 	test_size: 0.3


[run peachy-sweep-8] accuracy=0.9909, f1_macro=0.9897


f1_class_0,▁
f1_class_1,▁
f1_class_2,▁
f1_class_3,▁
f1_class_4,▁
f1_class_5,▁
val_accuracy,▁
val_f1_macro,▁
val_roc_auc_ovr,▁
f1_class_0,1
f1_class_1,0.97297


## What this lab demonstrated

In this updated lab we:

- Switched the model from **XGBoost** to a **Random Forest classifier**.
- Refactored the notebook into reusable functions (`load_dermatology_data`, `train_random_forest`).
- Used **W&B configs** to control model hyperparameters.
- Logged:
  - validation accuracy & macro F1
  - per-class F1-scores
  - confusion matrix
  - feature importance as a bar chart
  - a table of sample predictions
- Saved the trained model and logged it as a **W&B Artifact** for versioned model tracking.
- Ran a **W&B hyperparameter sweep** to automatically explore multiple Random Forest configurations.

This turns the original one-off demo into a real experiment-tracking workflow that you can reuse in larger ML projects.